# EchoAlign: Statistical Evaluation & AUROC Analysis

This notebook demonstrates the **cluster-bootstrap resampling protocol** used in the EchoAlign paper to calculate:
- Area Under the ROC Curve (AUROC) with 95% Confidence Intervals.
- Comparative performance analysis between EchoAlign and baselines (EchoPrime, PanEcho, EchoCLIP).

Resampling draws whole clusters (patients) with replacement, preserving within-patient correlation.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from eval.analysis.metrics import auroc_ci, cluster_draws, seed_from
from eval.analysis.data import load_tasks


## 1. Load Task Configurations

Inspect the zero-shot tasks and anatomical structure mappings across evaluation cohorts:

In [ ]:
tasks = load_tasks('../configs/zeroshot_tasks.yaml', 'mimic_iv_echo')
task_summary = []
for t, cfg in tasks.items():
    task_summary.append({
        'Task': t,
        'EchoAlign Structure': (cfg.get('echoalign') or {}).get('structure', 'global'),
        'PanEcho Output': (cfg.get('panecho') or {}).get('output', '-'),
        'EchoPrime Output': (cfg.get('echoprime') or {}).get('output', '-')
    })
df_tasks = pd.DataFrame(task_summary)
print(f'Total tasks configured: {len(df_tasks)}')
df_tasks.head(10)


## 2. Benchmark Evaluation Protocol (Cluster Bootstrap)

We generate synthetic predictions for 200 studies across 50 patients to show how AUROC and cluster-bootstrap 95% CIs are computed:

In [ ]:
rng = np.random.default_rng(42)
n_studies = 200
n_patients = 50

patients = [f'PATIENT_{i}' for i in rng.integers(0, n_patients, size=n_studies)]
y_true = rng.integers(0, 2, size=n_studies)
# Simulated model scores with positive correlation to ground truth
y_scores_echoalign = y_true * 0.4 + rng.normal(0, 0.3, size=n_studies)
y_scores_baseline = y_true * 0.25 + rng.normal(0, 0.35, size=n_studies)

# Compute cluster draws
uniq_patients, cluster_idx = np.unique(patients, return_inverse=True)
draws = cluster_draws(len(uniq_patients), replicates=1000, seed=seed_from('evaluation_demo'))

metrics_ev = auroc_ci(y_true, y_scores_echoalign, cluster_idx, draws)
metrics_base = auroc_ci(y_true, y_scores_baseline, cluster_idx, draws)

res_df = pd.DataFrame([
    {'Model': 'EchoAlign', 'AUROC': metrics_ev['auroc'], '95% CI Lower': metrics_ev['lower'], '95% CI Upper': metrics_ev['upper']},
    {'Model': 'Baseline', 'AUROC': metrics_base['auroc'], '95% CI Lower': metrics_base['lower'], '95% CI Upper': metrics_base['upper']}
])
res_df


## 3. Visualize AUROC Comparison

Plot the comparative performance with error bars representing the cluster-bootstrap 95% confidence intervals:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
yerr = [
    res_df['AUROC'] - res_df['95% CI Lower'],
    res_df['95% CI Upper'] - res_df['AUROC']
]
ax.bar(res_df['Model'], res_df['AUROC'], yerr=yerr, capsize=5, color=['#1f77b4', '#aec7e8'], edgecolor='black', alpha=0.85)
ax.set_ylabel('AUROC')
ax.set_ylim(0.5, 1.0)
ax.set_title('Model Performance with 95% Cluster-Bootstrap CI')
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
